<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/bonus/notebooks/12_text_as_data_for_causal_questions_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Bonus: Text as Data for Causal Questions

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

Most of the course measures things in text: frames, topics, sentiment, policy areas. Often the next
step is a **causal** question: did a news frame change what people think? Do conflict-framed articles
get shared more? Measuring with text in a causal analysis brings its own problems, and some of them
can quietly bias your estimates even when your classifier looks good.

In this notebook you will:

- see the three roles text can play in a causal analysis: outcome, treatment, confounder
- analyze a simulated framing experiment with an open-ended text outcome
- see how classifier errors bias treatment effects, and when they don't
- combine a classifier with a hand-coded random sample to get unbiased estimates (the idea behind
  **design-based supervised learning**, DSL)
- see why the same data shouldn't be used to discover a measure and to test it
- adjust for text as a confounder, and avoid adjusting for the wrong things

Everything is simulated, so we know the true effects and can check every estimator against them.

**Contents**

1. Three roles for text
2. A framing experiment with a text outcome
3. Measuring the outcome with a classifier
4. When classifier errors bias the effect
5. Fixing it with a hand-coded sample
6. Discovery and testing on the same data
7. Text as a confounder
8. Text as treatment, and a checklist

No GPU needed.

There are 5 exercises along the way, marked ✏️. A separate solutions notebook has worked answers, but try them yourself first.

## 0. Setup

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(2026)
pd.set_option("display.max_colwidth", 120)

## 1. Three roles for text

| Role | Example | Main danger |
|---|---|---|
| **Outcome** | Does reading an economic frame make people *write* more about economic consequences? | measurement error in the text-based outcome, especially if it differs between groups |
| **Treatment** | Which features of a petition *cause* more signatures? | discovering the "treatment" and testing it on the same data |
| **Confounder** | Do conflict-framed articles get shared more, *given* their topic? | adjusting for text that also contains the treatment, or that the treatment changed |

Causal inference with text uses the same logic as without it (randomization, or adjusting for
confounders), but the variables are *measured* from text, by a model. The model's errors and the
researcher's choices become part of the design.

## 2. A framing experiment with a text outcome

A survey experiment: 2,000 respondents are randomly assigned to read a news article about nitrogen
policy with either an **economic frame** (treatment, $T = 1$) or an **environmental frame** ($T = 0$).
Afterwards they answer an open question: *"What does this policy mean, in your own words?"*

Our outcome $Y$: does the answer discuss **economic consequences** (lost incomes, costs to taxpayers,
higher prices, a village that won't survive)? In the simulation, 30% of the control group do, and 42%
of the treated: a true effect of **12 percentage points**.

One realistic complication: some treated respondents **echo the article's words** ("the article was
all about costs and money") without discussing any economic consequences themselves. By the codebook,
that's not an economic consideration. Remember it; it matters in Section 4.

In [ ]:
def pick(options):
    return options[rng.integers(len(options))]


def economic_sentence():
    # Many ways of discussing economic consequences, some explicit, some subtle
    return pick([
        lambda: f"{pick(['farmers', 'families', 'young farmers', 'people in my village'])} will "
                f"{pick(['lose', 'struggle to keep'])} their {pick(['income', 'jobs', 'livelihood'])}",
        lambda: f"it will cost {pick(['taxpayers', 'us', 'the country'])} {pick(['a lot of money', 'billions', 'too much'])}",
        lambda: f"{pick(['food prices', 'prices in the supermarket', 'the price of milk'])} will go up",
        lambda: "my neighbor wonders how he will pay back the bank",
        lambda: "who is going to buy all that land",
        lambda: "the village will not survive without the dairy farms",
        lambda: "exports are what keeps the region going",
    ])()


OTHER = ["nature needs to be protected", "the heathland is dying because of nitrogen",
         "the government cannot be trusted on this", "politicians in The Hague do not listen",
         "my grandfather was a farmer too", "we need a plan for the next twenty years",
         "I hope the air and water get cleaner", "nature pays the price for intensive farming",
         "the protests went too far", "it is a difficult problem with no easy answer",
         "clean air is worth more than anything"]
ECHO = ["the article was all about costs and money", "they only talked about the economy",
        "costs, prices, jobs, that is what the text said", "it was about money again"]


def answer(y, t, echo=0.35):
    parts = list(rng.choice(OTHER, size=rng.integers(1, 3), replace=False))
    if y:
        parts.insert(rng.integers(0, len(parts) + 1), economic_sentence())
    elif t and rng.random() < echo:                     # treated respondents sometimes echo the frame
        parts.insert(0, pick(ECHO))
    text = ", ".join(parts)
    return text[0].upper() + text[1:] + "."


def run_experiment(n=2000, p0=0.30, effect=0.12, echo=0.35):
    t = rng.integers(0, 2, n)
    y = (rng.random(n) < p0 + effect * t).astype(int)
    return pd.DataFrame({"T": t, "Y": y, "text": [answer(yi, ti, echo) for yi, ti in zip(y, t)]})


exp = run_experiment()
print("True difference in this sample:", round(exp[exp["T"] == 1].Y.mean() - exp[exp["T"] == 0].Y.mean(), 3))
exp.sample(6, random_state=3)

In real life we don't know `Y`: we only have the texts. Coding 2,000 answers by hand is expensive, so we
measure `Y` automatically and hand-code a random sample to check (and, in Section 5, correct) the
automatic measure.

## 3. Measuring the outcome automatically

We use a simple **dictionary** of economic words (Day 1). It stands in for any automatic measure
that wasn't built on *this* data: a dictionary, a classifier trained on news articles, a zero-shot LLM
(Day 4). Everything below applies to all of them.

We also draw a **random 15%** of the answers (about 300) for hand-coding. In the simulation, hand-coding
simply reveals the true `Y`.

In [ ]:
ECON_WORDS = {"cost", "costs", "money", "economy", "economic", "price", "prices", "jobs", "income",
              "pay", "bank", "afford", "billions", "exports", "livelihood", "taxpayers"}


def dictionary(text):
    return int(bool(ECON_WORDS & set(re.findall(r"[a-z]+", text.lower()))))


PI = 0.15                                              # probability of being hand-coded


def draw_coding_sample(df, pi=PI, seed=0):
    df = df.copy()
    df["coded"] = np.random.default_rng(seed).random(len(df)) < pi
    return df


exp["Y_pred"] = exp.text.map(dictionary)
exp = draw_coding_sample(exp)
coded = exp[exp.coded]
print(f"{len(coded)} answers hand-coded")
print(f"Dictionary accuracy on the hand-coded answers: {(coded.Y_pred == coded.Y).mean():.3f}")

Not great, but typical for a simple dictionary. Now the obvious analysis: compare the share of answers
the dictionary flags as economic between the two groups.

In [ ]:
def effect(df, outcome):
    # Difference in means via OLS, with robust (HC2) standard errors
    fit = smf.ols(f"{outcome} ~ T", data=df).fit(cov_type="HC2")
    lo, hi = fit.conf_int().loc["T"]
    return {"estimate": fit.params["T"], "se": fit.bse["T"], "ci_low": lo, "ci_high": hi}


results = pd.DataFrame({
    "truth (unknown in practice)": effect(exp, "Y"),
    "dictionary, all 2,000": effect(exp, "Y_pred"),
}).T
results.round(3)

## 4. When measurement errors bias the effect

The dictionary-based estimate is far too large. Overall accuracy doesn't tell you why; the error rates
**per group** do, and you can compute them from the hand-coded sample:

In [ ]:
def error_rates(d):
    return pd.Series({"false positive rate": d[d.Y == 0].Y_pred.mean(),
                      "false negative rate": 1 - d[d.Y == 1].Y_pred.mean(), "n coded": len(d)})

coded.groupby("T").apply(error_rates, include_groups=False).round(3)

The false positive rate is much higher in the treatment group: the dictionary picks up the echoed
words "costs", "money" and "economy". The errors are **differential**: they depend on the treatment.
That adds a spurious difference between the groups.

Two kinds of measurement error in an outcome:

- **Non-differential** (same error rates in both groups): for a binary outcome, the effect is
  *attenuated*, shrunk toward zero by a factor $1 - \text{FPR} - \text{FNR}$. You may miss a real
  effect, but you won't invent one.
- **Differential** (error rates differ by group): the bias can go **either way**, and can be large,
  even with a measure that's quite accurate overall.

The treatment can easily change the *language* of the outcome texts (vocabulary from the stimulus,
answer length, tone), so differential error is the realistic case in text experiments. A classifier
trained on the experiment's own hand-coded answers suffers less from it, but you can only know that
by checking.

> **✏️ Exercise 1**
>
> Explore the two kinds of error without a dictionary. Take the true `Y` from `exp` and create noisy
> versions:
>
> 1. **non-differential**: flip each 0 to 1 with probability 0.10 and each 1 to 0 with probability
>    0.10, in both groups;
> 2. **differential**: flip 0 to 1 with probability 0.20 in the treatment group and 0.05 in the control
>    group (and 1 to 0 with probability 0.10 in both).
>
> Estimate the effect with each. Compare with the truth and with the attenuation formula
> $(1 - 0.10 - 0.10) \times$ true effect.

In [ ]:
# Your code here

## 5. Fixing it with a hand-coded sample

We have about 300 answers with a known (hand-coded) outcome, drawn **at random**. Two ways to use them:

**Gold only.** Estimate the effect on the hand-coded answers alone. Unbiased (randomization within a
random sample), but noisy: 300 instead of 2,000 observations.

**Design-based supervised learning (DSL)** (Egami et al., 2023) uses both. For every answer, it builds
a corrected outcome:

$$\tilde{Y}_i = \hat{Y}_i + \frac{R_i}{\pi_i}\left(Y_i - \hat{Y}_i\right)$$

where $\hat{Y}_i$ is the automatic measure, $R_i = 1$ if the answer was hand-coded, and $\pi_i$ the
known probability of being hand-coded (0.15 here). For answers that weren't hand-coded,
$\tilde{Y}_i = \hat{Y}_i$; for hand-coded ones, the measurement error is added back, weighted up by
$1/\pi_i$ so that the 300 stand in for everyone. Because the hand-coded sample is random, the
corrections cancel the measure's errors *on average*, whatever those errors look like, differential
or not. Then we estimate the effect on $\tilde{Y}$ as usual.

Requirements: the hand-coding sample must be drawn with **known probabilities** (simple random
sampling is easiest), and if $\hat{Y}$ comes from a classifier *trained* on the hand-coded answers,
their predictions must come from a model that didn't see them (cross-fitting; Exercise 3).

In [ ]:
def dsl_outcome(df, pi=PI, pred="Y_pred"):
    return df[pred] + df.coded / pi * (df.Y.where(df.coded, 0) - df[pred])

exp["Y_dsl"] = dsl_outcome(exp)

results = pd.DataFrame({
    "truth (unknown in practice)": effect(exp, "Y"),
    "dictionary, all 2,000": effect(exp, "Y_pred"),
    "gold only, ~300": effect(exp[exp.coded], "Y"),
    "DSL, all 2,000": effect(exp, "Y_dsl"),
}).T
results.round(3)

In one sample, any estimator can be lucky or unlucky. The real test is many repetitions: the next
exercise draws the hand-coding sample 500 times.

(The `Y.where(df.coded, 0)` only avoids using `Y` for answers that weren't hand-coded; their term is
multiplied by 0 anyway. In a real project, `Y` simply doesn't exist for them.)

> **✏️ Exercise 2**
>
> Draw the hand-coding sample 500 times (use `draw_coding_sample(exp, seed=s)`), and each time estimate
> the effect with the dictionary, gold only, and DSL. For each estimator, report the average estimate
> (bias = average minus the true effect in `exp`), the standard deviation across repetitions, and the
> share of 95% confidence intervals that contain the true effect (coverage). Which would you use?

In [ ]:
# Your code here

> **✏️ Exercise 3**
>
> DSL is more precise when $\hat{Y}$ is better. Instead of the dictionary, train a classifier on the
> hand-coded answers, using **cross-fitting**: for the hand-coded answers, cross-validated predictions;
> for the rest, a model trained on all hand-coded answers. The function below does that. Over 100
> draws of the hand-coding sample, compare bias and standard deviation of DSL with the dictionary and
> DSL with the trained classifier.
>
> ```python
> def classifier_predictions(df):
>     model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2), LogisticRegression(C=3, max_iter=1000))
>     c = df[df.coded]
>     pred = pd.Series(0.0, index=df.index)
>     pred[df.coded] = cross_val_predict(model, c.text, c.Y, cv=5, method="predict_proba")[:, 1]
>     model.fit(c.text, c.Y)
>     pred[~df.coded] = model.predict_proba(df.loc[~df.coded, "text"])[:, 1]
>     return pred
> ```

In [ ]:
# Your code here

## 6. Discovery and testing on the same data

With text, the measure itself is often **discovered** from the data: you read answers, build a
dictionary of words that seem to separate the groups, or fit a topic model and pick an interesting
topic. If you then test the effect on the *same* answers, you're testing a measure tailored to this
sample's quirks, including pure chance. Egami et al. (2022) recommend a **split sample**: discover on
one part, test on the other, as if pre-registering the measure.

The demonstration: an experiment in which the treatment has **no effect at all**. We "discover" the
words most over-represented in the treatment group and test whether answers containing them differ
between groups.

In [ ]:
null_exp = run_experiment(effect=0.0, echo=0.0)        # no effect and no echo: T changes nothing
counts = CountVectorizer(min_df=5, binary=True).fit(null_exp.text)
X = counts.transform(null_exp.text).toarray()
vocab = counts.get_feature_names_out()


def discover_and_test(X, t, discover_idx, test_idx, k=5):
    # Find the k words most over-represented among treated in discover_idx, then test in test_idx
    d = X[discover_idx]
    diff = d[t[discover_idx] == 1].mean(axis=0) - d[t[discover_idx] == 0].mean(axis=0)
    words = diff.argsort()[::-1][:k]
    score = (X[test_idx][:, words].sum(axis=1) > 0).astype(int)
    fit = smf.ols("score ~ T", data=pd.DataFrame({"score": score, "T": t[test_idx]})).fit(cov_type="HC2")
    return vocab[words], fit.params["T"], fit.pvalues["T"]


t = null_exp["T"].values
idx = rng.permutation(len(t))
half_a, half_b = idx[: len(t) // 2], idx[len(t) // 2:]
words, est, p = discover_and_test(X, t, half_a, half_a)
print("Discovered words:", list(words))
print(f"Tested on the SAME half:  effect {est:+.3f}, p = {p:.3f}")
words, est, p = discover_and_test(X, t, half_a, half_b)
print(f"Tested on the OTHER half: effect {est:+.3f}, p = {p:.3f}")

> **✏️ Exercise 4**
>
> One split can be lucky. Repeat the procedure 200 times with new random splits, and record how
> often the test is "significant" (p < 0.05) when (a) discovering and testing on the same half, and
> (b) testing on the other half. With no true effect, what share would you expect?

In [ ]:
# Your code here

## 7. Text as a confounder

An observational question: do **conflict-framed** news articles get **shared more**? The problem:
political articles are both more often conflict-framed *and* more often shared, regardless of frame.
Topic is a **confounder**, and it's in the text.

The simulation: 3,000 articles about politics, local news or science. Conflict framing is more common in
politics. Shares (on a log scale) depend on the topic and on the frame; the true effect of the
conflict frame is **0.40**.

In [ ]:
TOPIC_WORDS = {"politics": "minister parliament coalition election party vote cabinet senate policy",
               "local": "municipality council village road school festival neighbors library park",
               "science": "researchers study university climate data experiment species measured journal"}
CONFLICT_WORDS = "clash attack fight slams furious battle row accuses"
FILLER = "the on in a and of to with this week new said report people today city about year"


def article(topic, conflict):
    words = (rng.choice(TOPIC_WORDS[topic].split(), size=8).tolist()
             + rng.choice(FILLER.split(), size=10).tolist()
             + (rng.choice(CONFLICT_WORDS.split(), size=3).tolist() if conflict else []))
    rng.shuffle(words)
    return " ".join(words)


n = 3000
topic = rng.choice(["politics", "local", "science"], size=n, p=[0.40, 0.35, 0.25])
conflict = (rng.random(n) < pd.Series(topic).map({"politics": 0.6, "local": 0.3, "science": 0.15})).astype(int)
log_shares = (2 + pd.Series(topic).map({"politics": 1.0, "local": 0.3, "science": 0.0})
              + 0.40 * conflict + rng.normal(0, 0.8, n))
news = pd.DataFrame({"topic": topic, "T": conflict, "log_shares": log_shares,
                     "text": [article(tp, c) for tp, c in zip(topic, conflict)]})


def frame_effect(formula, df):
    fit = smf.ols(formula, data=df).fit(cov_type="HC2")
    return round(float(fit.params["T"]), 3), round(float(fit.bse["T"]), 3)


print("naive (no adjustment):         ", frame_effect("log_shares ~ T", news))
print("adjusted for the TRUE topic:   ", frame_effect("log_shares ~ T + C(topic)", news))

In practice, the topic isn't a column in your data; you estimate it from the text, for example with a
topic model (Bonus 11) or, here, non-negative matrix factorization (NMF) on TF-IDF, a quick relative of
topic models. Then you adjust for the estimated topic proportions.

There's a trap: if the text representation also captures the **treatment** (here: the conflict
words), adjusting for it absorbs part of the treatment itself. The confounder must be measured from text
*without* the treatment.

> **✏️ Exercise 5**
>
> 1. Fit NMF with 3 components on TF-IDF of **all** words, add the three topic proportions as
>    covariates (`log_shares ~ T + nmf0 + nmf1 + nmf2`), and estimate the effect.
> 2. Do the same, but remove the conflict words from the vocabulary first
>    (`TfidfVectorizer(stop_words=CONFLICT_WORDS.split())`).
> 3. Finally, a **post-treatment** variable: suppose conflict-framed articles more often get a
>    sensational headline (`sensational = rng.random(n) < 0.2 + 0.5 * T`), and sensational headlines
>    add 0.3 to log shares. Add this to `log_shares`, then compare the effect of `T` with and without
>    adjusting for `sensational`. The *total* effect of the frame is now 0.40 + 0.5 × 0.3 = 0.55.

In [ ]:
# Your code here

## 8. Text as treatment, and a checklist

### Text as treatment

Which features of a text *cause* a response? Which words in a petition attract signatures, which
headline gets clicks, which arguments persuade? Two routes:

- **Randomize texts.** Write versions that differ in one feature and assign them at random (survey
  experiments, A/B tests). The cleanest design; the hard part is writing versions that differ *only*
  in the feature of interest.
- **Discover treatments from observational texts** (Fong & Grimmer, 2016). Here the split-sample logic
  of Section 6 is essential: discover candidate features in one part, estimate their effects in the
  other, and remember that texts bundle many features that are hard to separate.

### Checklist

- Is text your **outcome, treatment or confounder**? Each has its own pitfalls.
- **Validate your measure per group** (per treatment condition, per period, per source), not only
  overall: differential error is the dangerous kind.
- Hand-code a **random sample with known probabilities**, and use it in the estimation (DSL or
  similar), not just to report accuracy.
- Don't **discover and test** on the same data. Split, or pre-register the measure.
- When adjusting for text, make sure the representation **excludes the treatment**, and **don't adjust
  for post-treatment** text features.
- LLM-based measures (Day 4) are classifiers too: everything here applies to them.

### Further reading

- Egami, N., Fong, C. J., Grimmer, J., Roberts, M. E., & Stewart, B. M. (2022). How to make causal
  inferences using texts. *Science Advances, 8*(42), eabg2652.
- Egami, N., Hinck, M., Stewart, B. M., & Wei, H. (2023). Using imperfect surrogates for downstream
  inference: Design-based supervised learning for social science applications of large language
  models. *Advances in Neural Information Processing Systems 36* (NeurIPS 2023). (R package: `dsl`.)
- Feder, A., Keith, K. A., Manzoor, E., et al. (2022). Causal inference in natural language processing: Estimation, prediction, interpretation and beyond.
  *Transactions of the ACL, 10*, 1138-1158.
- Fong, C., & Grimmer, J. (2016). Discovery of treatments from text corpora. *Proceedings of ACL 2016*.
- Grimmer, J., Roberts, M. E., & Stewart, B. M. (2022). *Text as data: A new framework for machine
  learning and the social sciences*. Princeton University Press.
- Keith, K. A., Jensen, D., & O'Connor, B. (2020). Text and causal inference: A review of using text to
  remove confounding from causal estimates. *Proceedings of ACL 2020*.
- Roberts, M. E., Stewart, B. M., & Nielsen, R. A. (2020). Adjusting for confounding with text
  matching. *American Journal of Political Science, 64*(4), 887-903.